#### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function to co-routine to execute

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("What is openclaw?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "What is openclaw?"\n   - The term "openclaw" doesn\'t immediately ring a bell as a widely known technology, framework, or concept. I need to verify what it refers to.\n\n2.  **Initial Knowledge Retrieval & Reasoning:**\n   - Could it be a typo? Maybe "OpenCL"? OpenCL is a well-known framework for parallel programming across heterogeneous platforms.\n   - Could it be "OpenClaw" as a specific project, library, or tool? I should check if it\'s an emerging/open-source project.\n   - Could it be related to gaming, AI, security, or something else?\n   - Let\'s think about possible interpretations:\n     - "OpenCL" + "aw"? Unlikely.\n     - Maybe "Open Claw" as a hardware/software project?\n     - Could be a niche GitHub repo or startup.\n     - Might be a misspelling of "OpenCV", "OpenCL", "OpenSSL", etc.\n   - I should acknowledge uncertainty, ask for clarification, and provide possibl

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What is the weather in Kolkata?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user wants to know the weather in Kolkata.\n2.  Identify available tools: `get_weather(location)`\n3.  Extract parameters: `location` = "Kolkata"\n4.  Call the tool: `get_weather(location="Kolkata")`\n5.  Process the response and formulate the answer. (I will simulate the tool call and response generation)\nLet\'s call the tool. \nWait, I need to output the tool call exactly as specified.\n`get_weather(location="Kolkata")`\nProceed. \nSelf-Correction/Verification: The tool schema requires `location` as a string. I have it. I will generate the function call. \nOutput matches the required format.✅\nProceeds. \n[Output Generation] -> `call: default_api:get_weather{"location": "Kolkata"}` (or similar format depending on system prompt, but I\'ll stick to the standard function calling format).\nActually, the prompt says: "If you choose to call a function ONLY reply in the following for

#### Tool Execution Loops

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "what is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It is sunny in Boston.
